In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
def get_y_counts(df, digits=3, col="Y_haplo"):
    """Get Y Chromosome counts from Dataframe df"""
    ys = df[col].str[:3]
    cts = ys.value_counts().values
    return cts
def get_y_counts_8(df, digits=8, col="Y_haplo"):
    """Get Y Chromosome counts from Dataframe df"""
    ys = df[col].str[:8]
    cts = ys.value_counts().values
    return cts
def simpson_di(x):
    """ Given a count vector, returns the Simpson Diversity Index
    """
    n = np.sum(x) # Sample Size
    h = np.sum(x*(x-1)) / (n*(n-1)) # Fraction of pairs are identiclal
    
    if h==0: ### Set minimimum homo-cutoff (one homo-pair):
        h = 2 / (n*(n-1))
    return 1 / h

In [ ]:
Ymeta=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/Hap/YHap_150925.csv',names=['ID','Periodiation','Genetic_Sex','MT_Haplogroup','Haplogrep_quality','Mtcov_schmutzi','Y_haplo']
                ,skiprows=1)

In [ ]:
import pandas as pd

def get_y_counts_8(df, digits=8, col="Y_haplo"):
    """Get Y Chromosome counts from DataFrame df, filtering out entries with fewer than `digits` characters."""
    # Filter out entries with insufficient length
    df_filtered = df[df[col].str.len() >= digits].copy()
    
    # Take the first `digits` characters
    df_filtered['Y_prefix'] = df_filtered[col].str[:digits]
    
    # Count frequencies
    counts_df = df_filtered['Y_prefix'].value_counts().reset_index()
    counts_df.columns = ['Y_prefix', 'count']
    
    return counts_df

# Subset data by period
Ymeta_LBA = Ymeta[Ymeta['Periodiation'] == 'KNC_LBA']
Ymeta_EIA = Ymeta[Ymeta['Periodiation'] == 'KNC_EIA']
Ymeta_IA  = Ymeta[Ymeta['Periodiation'] == 'KNC_IA']

# Get counts
cts_LBA_df = get_y_counts_8(Ymeta_LBA)
cts_EIA_df = get_y_counts_8(Ymeta_EIA)
cts_IA_df  = get_y_counts_8(Ymeta_IA)

# Print counts
print("LBA Counts:\n", cts_LBA_df)
print("EIA Counts:\n", cts_EIA_df)
print("IA Counts:\n", cts_IA_df)

# Extract only the count values for Simpson index calculation
cts_LBA = cts_LBA_df['count'].values
cts_EIA = cts_EIA_df['count'].values
cts_IA  = cts_IA_df['count'].values

# Calculate Simpson diversity indices
SimInd_LBA = simpson_di(cts_LBA)
SimInd_EIA = simpson_di(cts_EIA)
SimInd_IA  = simpson_di(cts_IA)

# Print results
print("Simpson Index LBA:", SimInd_LBA)
print("Simpson Index EIA:", SimInd_EIA)
print("Simpson Index IA:", SimInd_IA)


In [ ]:
import numpy as np

def simpson_di(counts):
    """Calculate Simpson's Diversity Index from count array."""
    counts = np.array(counts)
    N = counts.sum()
    if N <= 1:
        return 0
    return 1 / (np.sum(counts * (counts - 1)) / (N * (N - 1)))

def bootstrap_simpson(counts, n_boot=10000):
    """Bootstrap distribution of Simpson index."""
    counts = np.array(counts)
    total = counts.sum()
    values = []

    for _ in range(n_boot):
        # Generate random sample with replacement
        sample = np.random.choice(np.repeat(np.arange(len(counts)), counts), size=total, replace=True)
        boot_counts = np.bincount(sample, minlength=len(counts))
        values.append(simpson_di(boot_counts))

    return np.array(values)

# Example usage
counts_LBA = np.array([8, 3, 2])   # Replace with actual counts
counts_EIA = np.array([29, 15, 4, 2, 2, 2, 1])

# Bootstrap both
boot_LBA = bootstrap_simpson(counts_LBA)
boot_EIA = bootstrap_simpson(counts_EIA)

# Calculate difference
diff = boot_LBA - boot_EIA
p_value = (np.sum(diff > 0) / len(diff)) * 2  # two-tailed test

print(f"Mean LBA Simpson: {boot_LBA.mean():.3f}")
print(f"Mean EIA Simpson: {boot_EIA.mean():.3f}")
print(f"P-value for difference: {p_value:.4f}")


In [ ]:
import numpy as np

def simpson_di(counts):
    """Calculate Simpson's Diversity Index from count array."""
    counts = np.array(counts)
    N = counts.sum()
    if N <= 1:
        return 0
    return 1 / (np.sum(counts * (counts - 1)) / (N * (N - 1)))

def bootstrap_simpson(counts, n_boot=10000):
    """Bootstrap distribution of Simpson index."""
    counts = np.array(counts)
    total = counts.sum()
    values = []

    for _ in range(n_boot):
        # Generate random sample with replacement
        sample = np.random.choice(np.repeat(np.arange(len(counts)), counts), size=total, replace=True)
        boot_counts = np.bincount(sample, minlength=len(counts))
        values.append(simpson_di(boot_counts))

    return np.array(values)

# Example usage

counts_EIA = np.array([29, 15, 4, 2, 2, 2, 1])
counts_DIA = np.array([31, 5])   # Replace with actual counts

# Bootstrap both
boot_DIA = bootstrap_simpson(counts_DIA)
boot_EIA = bootstrap_simpson(counts_EIA)

# Calculate difference
diff = boot_DIA - boot_EIA
p_value = 2 * min(
    np.mean(diff >= 0),
    np.mean(diff <= 0)
)
  # two-tailed test

print(f"Mean DIA Simpson: {boot_DIA.mean():.3f}")
print(f"Mean EIA Simpson: {boot_EIA.mean():.3f}")
print(f"P-value for difference: {p_value:.10f}")
